# 03_map_visualization.ipynb - Visualisasi Peta Distribusi

Notebook ini untuk:
- Visualisasi sebaran outlet pada peta Indonesia
- Plot GPS tracking sales
- Heatmap ghost outlet & churn risk
- Interactive map dengan Folium

**ATURAN:** Notebook ini idempotent - bisa dijalankan ulang dari awal.

In [ ]:
# ── SETUP ────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, MarkerCluster

from config import DATA_DIR, OUT_DIR, REF_DIR
from geo_validator import parse_coords, clean_coords

print("✅ Setup complete")

In [ ]:
# ── LOAD DATA ────────────────────────────────────────────────────────────
print("📂 Loading data...")

df_address = pd.read_parquet(DATA_DIR / "address.parquet")
df_doccall = pd.read_parquet(DATA_DIR / "doccall.parquet")
df_gps = pd.read_parquet(DATA_DIR / "gps.parquet")

# Load hasil analisis dari notebook 02
ghost_outlets = pd.read_csv(OUT_DIR / "ghost_outlets.csv")
sales_perf = pd.read_csv(OUT_DIR / "sales_performance.csv")

print(f"  ✅ address: {len(df_address):,} rows")
print(f"  ✅ doccall: {len(df_doccall):,} rows")
print(f"  ✅ gps: {len(df_gps):,} rows")
print(f"  ✅ ghost_outlets: {len(ghost_outlets):,} rows")
print(f"  ✅ sales_performance: {len(sales_perf):,} rows")

In [ ]:
# ── PARSE KOORDINAT ─────────────────────────────────────────────────────
# Rule #3 & #4: Handle szLangitude (TYPO) dan szLatitude

print("📍 Parsing coordinates...")

# Parse koordinat dari address
df_address['lat'] = df_address['szLatitude'].apply(parse_coords)
df_address['lon'] = df_address['szLongitude'].apply(parse_coords)

# Filter koordinat valid
df_address_clean = clean_coords(df_address, lat_col='lat', lon_col='lon')

print(f"  Total outlet: {len(df_address):,}")
print(f"  Dengan koordinat valid: {len(df_address_clean):,}")
print(f"  Invalid/Null coords: {len(df_address) - len(df_address_clean):,}")

In [ ]:
# ── PETA 1: SEBARAN SEMUA OUTLET ────────────────────────────────────────
print("\n🗺️  Creating map: All Outlet Distribution...")

# Center map di Indonesia
m_all = folium.Map(location=[-2.5, 118], zoom_start=5, tiles='OpenStreetMap')

# Tambahkan marker cluster untuk performa
marker_cluster = MarkerCluster().add_to(m_all)

# Sample data jika terlalu banyak (max 1000 marker untuk performa)
df_sample = df_address_clean.sample(min(1000, len(df_address_clean)))

for idx, row in df_sample.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=3,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.6,
        popup=f"{row['szCustomerName']}<br>{row['szCity']}",
        tooltip=row['szCustomerName']
    ).add_to(marker_cluster)

# Simpan map
m_all.save(OUT_DIR / "map_all_outlets.html")
print(f"  ✅ Saved: map_all_outlets.html")

# Tampilkan map (hanya jika di Jupyter)
m_all

In [ ]:
# ── PETA 2: GHOST OUTLET HEATMAP ────────────────────────────────────────
print("\n🗺️  Creating map: Ghost Outlet Heatmap...")

# Merge ghost outlets dengan address
df_ghost = ghost_outlets.merge(df_address[['szCustomerId', 'lat', 'lon']], 
                                on='szCustomerId', how='left')
df_ghost = df_ghost.dropna(subset=['lat', 'lon'])

m_ghost = folium.Map(location=[-2.5, 118], zoom_start=5, tiles='CartoDB dark_matter')

# Buat heatmap
heat_data = df_ghost[['lat', 'lon']].values.tolist()
HeatMap(heat_data, radius=15, blur=20, max_zoom=1).add_to(m_ghost)

# Tambahkan layer kontrol
folium.LayerControl().add_to(m_ghost)

m_ghost.save(OUT_DIR / "map_ghost_outlets.html")
print(f"  ✅ Saved: map_ghost_outlets.html")
print(f"  📊 Ghost outlets plotted: {len(df_ghost):,}")

m_ghost

In [ ]:
# ── PETA 3: CHURN RISK OUTLET ───────────────────────────────────────────
print("\n🗺️  Creating map: Churn Risk Outlets...")

# Filter churn risk outlets
df_churn = sales_perf[sales_perf['churn_risk'] == True].merge(
    df_address[['szCustomerId', 'lat', 'lon', 'szCity']], 
    on='szCustomerId', how='left'
)
df_churn = df_churn.dropna(subset=['lat', 'lon'])

m_churn = folium.Map(location=[-2.5, 118], zoom_start=5, tiles='OpenStreetMap')

# Plot churn risk dengan warna merah
for idx, row in df_churn.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color='red',
        fill=True,
        fill_color='red',
        fill_opacity=0.7,
        popup=f"{row['szCustomerName']}<br>Churn Risk: {row['sales_change_pct']*100:.1f}%<br>{row['szCity']}",
        tooltip=f"{row['szCustomerName']} ({row['sales_change_pct']*100:.0f}%)"
    ).add_to(m_churn)

m_churn.save(OUT_DIR / "map_churn_risk.html")
print(f"  ✅ Saved: map_churn_risk.html")
print(f"  📊 Churn risk outlets: {len(df_churn):,}")

m_churn

In [ ]:
# ── PETA 4: GPS TRACKING SALES (Sample) ─────────────────────────────────
print("\n🗺️  Creating map: GPS Tracking Sales...")

# Parse koordinat GPS
# Rule #3: Kolom bernama szLangitude (TYPO!)
df_gps['lat'] = df_gps['szLangitude'].apply(parse_coords)
df_gps['lon'] = df_gps['szLongitude'].apply(parse_coords)
df_gps_clean = clean_coords(df_gps, lat_col='lat', lon_col='lon')

# Group by sales employee
sales_ids = df_gps_clean['szEmployeeId'].unique()
print(f"  Total sales dengan GPS: {len(sales_ids)}")

# Ambil sample 10 sales pertama
sample_sales = sales_ids[:10]

m_gps = folium.Map(location=[-6.2, 106.8], zoom_start=10, tiles='OpenStreetMap')

colors = ['blue', 'green', 'purple', 'orange', 'darkred', 
          'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue']

for i, emp_id in enumerate(sample_sales):
    df_emp = df_gps_clean[df_gps_clean['szEmployeeId'] == emp_id].sort_values('dtTimestamp')
    
    if len(df_emp) < 2:
        continue
    
    # Plot route line
    coordinates = df_emp[['lat', 'lon']].values.tolist()
    folium.PolyLine(
        locations=coordinates,
        color=colors[i % len(colors)],
        weight=3,
        opacity=0.7,
        tooltip=f"Sales: {emp_id}"
    ).add_to(m_gps)
    
    # Tambahkan marker start dan end
    folium.Marker(
        location=coordinates[0],
        icon=folium.Icon(color='green', icon='play'),
        tooltip=f"Start: {emp_id}"
    ).add_to(m_gps)
    
    folium.Marker(
        location=coordinates[-1],
        icon=folium.Icon(color='red', icon='stop'),
        tooltip=f"End: {emp_id}"
    ).add_to(m_gps)

m_gps.save(OUT_DIR / "map_gps_tracking.html")
print(f"  ✅ Saved: map_gps_tracking.html")
print(f"  📊 Sales routes plotted: {len(sample_sales)}")

m_gps

In [ ]:
# ── PETA 5: KUNJUNGAN SALES PER OUTLET ──────────────────────────────────
print("\n🗺️  Creating map: Visit Frequency...")

# Hitung frekuensi kunjungan per outlet
visit_counts = df_doccall.groupby('szCustomerId').size().reset_index(name='visit_count')

# Merge dengan address
df_visits = visit_counts.merge(df_address[['szCustomerId', 'lat', 'lon', 'szCustomerName']], 
                                on='szCustomerId', how='left')
df_visits = df_visits.dropna(subset=['lat', 'lon'])

m_visits = folium.Map(location=[-2.5, 118], zoom_start=5, tiles='OpenStreetMap')

# Warna berdasarkan frekuensi kunjungan
def get_color(count):
    if count >= 10:
        return 'green'
    elif count >= 5:
        return 'orange'
    else:
        return 'red'

for idx, row in df_visits.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=min(10, row['visit_count'] / 2),
        color=get_color(row['visit_count']),
        fill=True,
        fill_color=get_color(row['visit_count']),
        fill_opacity=0.6,
        popup=f"{row['szCustomerName']}<br>Visits: {row['visit_count']}",
        tooltip=f"{row['visit_count']} visits"
    ).add_to(m_visits)

m_visits.save(OUT_DIR / "map_visit_frequency.html")
print(f"  ✅ Saved: map_visit_frequency.html")
print(f"  📊 Outlets with visits: {len(df_visits):,}")

m_visits

In [ ]:
# ── RINGKASAN ───────────────────────────────────────────────────────────
print("\n" + "="*60)
print("✅ MAP VISUALIZATION COMPLETE")
print("="*60)
print(f"\n📍 Maps saved to {OUT_DIR}:")
print("   1. map_all_outlets.html - Sebaran semua outlet")
print("   2. map_ghost_outlets.html - Heatmap ghost outlet")
print("   3. map_churn_risk.html - Outlet churn risk")
print("   4. map_gps_tracking.html - GPS tracking sales")
print("   5. map_visit_frequency.html - Frekuensi kunjungan")
print("\n💡 Buka file HTML di browser untuk interactive map!")
print("\n🚀 Lanjut ke notebook 04_export_report.ipynb")